In [ ]:
# Install Prereqs
%pip install -qU langchain-text-splitters
%pip install -qU langchain-ollama
%pip install langchain.
%pip install ollama  

In [ ]:
# Step 1: Chunking

# I want to try Recursive Character Splitting via
# https://www.reddit.com/r/Rag/comments/1r47duk/we_benchmarked_7_chunking_strategies_most_best/

# https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

dataset = []

# TODO: Change these parameters later
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

# TODO: Be able to take in PDFs/other media files.
with os.scandir('sample-data') as es:
    for e in es:
        if e.is_file() and e.name.endswith('.md'):
            with open(e.path, 'r', encoding='utf-8') as file:
                file_metadata = {'filename': file.name} # We need the file name to source it later on.
                dataset.extend(text_splitter.create_documents(texts=[file.read()], metadatas=[file_metadata])) # Extend instead of append

dataset

[Document(metadata={'filename': 'sample-data\\advisory-group-members.md'}, page_content='---\nlayout: default\ntitle: Advisory Group Members\ndescription: "This page lists the members of the Diversity, Inclusion & Belonging Advisory Group."\n---\n\n{{< group-by-expertise "Diversity, Inclusion & Belonging Advisory Group" >}}'),
 Document(metadata={'filename': 'sample-data\\being-an-ally.md'}, page_content='---\ntitle: "The Ally Lab"\ndescription: Learn what is an ally, how to be an ally and what it means to be an ally.\n---\n\n## What is an ally?\n\nA diversity, inclusion and belonging "ally" is someone who is willing to take action in support of another person, in order to remove barriers that impede that person from contributing their skills and talents in the workplace or community.\n\nBeing an ally is a verb, this means that you proactively and purposefully take action and is not something forced upon you.'),
 Document(metadata={'filename': 'sample-data\\being-an-ally.md'}, page_con

In [69]:
# Step 2: Embedding, Vector Database and Retrieval

import numpy as np
from langchain_ollama import OllamaEmbeddings

# https://reference.langchain.com/python/langchain-core/embeddings/embeddings/Embeddings
# https://docs.langchain.com/oss/python/integrations/embeddings/ollama 

# I'm using llama3 since its local
# TODO: Explore different embedding models for deployment
EMBEDDING_MODEL = 'llama3'

embeddings = OllamaEmbeddings (
    model=EMBEDDING_MODEL,
)

from langchain_core.vectorstores import InMemoryVectorStore

# Vector database from all the documents embedded
# TODO: Consider FAISS?
vectorstore = InMemoryVectorStore.from_documents(
    documents=[dataset[i] for i in range(len(dataset))],
    embedding=embeddings,
)

# Set vectorstore as the retriever
# https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever
retriever = vectorstore.as_retriever()

# Retrieve the most similar text
retrieved_documents = retriever.invoke("What is an ally?")

print(retrieved_documents[0].page_content)
# That's not a bad result!

To be an effective ally it is important to understand some of the skills and behaviors great allies exhibit.


In [70]:
# Step 3: Generation

from langchain_ollama import OllamaLLM

# TODO: Change these parameters
model = OllamaLLM(
    model = 'llama3',
    temperature = 0.5, 
    num_predict = 256,
)

input_text = 'What is an ally?'
response = model.invoke(input_text)
print(response)
# Example without RAG prompting

An ally is a person, group, or organization that provides support and solidarity to another individual, group, or community that shares a common goal, interest, or identity. Allies often work together with the marginalized or oppressed group to promote their rights, interests, and well-being.

In various contexts, allies can take on different roles:

1. **Social justice ally**: An ally in social justice movements, such as feminism, anti-racism, LGBTQ+ rights, or disability advocacy, who actively supports and amplifies the voices of marginalized communities.
2. **Mentor or role model**: An ally who provides guidance, encouragement, and inspiration to individuals from underrepresented groups, helping them to navigate challenges and achieve their goals.
3. **Advocate**: An ally who uses their voice, influence, and resources to promote the interests and needs of a marginalized group, often in areas such as education, employment, or healthcare.
4. **Partner or collaborator**: An ally who wo

In [71]:
# Step 4: Prompting

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# TODO: Prompt adjustments
prompt = ChatPromptTemplate.from_template(""" Answer using ONLY the context below. If the context doesn't contain the answer,
then mention that it doesn't. Cite the source file from the metadata as ONLY a file name ONLY in a section of its own.

Context:
{context}

Question: {question}
""")

RAG = (
    {
    'context': retriever,
    'question': RunnablePassthrough()
    }
    | prompt
    | model
)

answer = RAG.invoke('What is an ally?')
print(answer)
# Looks pretty good!
# My biggest pet peave here is that it'll cite the source elsewhere as well- but I only want it to cite it at the end.

According to the context, an ally is someone who is willing to take action in support of another person, in order to remove barriers that impede that person from contributing their skills and talents in the workplace or community.

Source file: being-an-ally.md
